# 06 - Hydration Signal Model (EfficientNet-B0)

Train a Gabor + LBP hydration scorer for the **hydration** skin signal.

## Datasets
- **CelebA** - skin smoothness attributes as hydration proxy
- **Oulu Skin Condition Dataset** - skin condition labels with moisture annotations

## Features
- Gabor filter bank (4 orientations x 3 frequencies) for texture analysis
- LBP (Local Binary Pattern) histograms for micro-texture
- Specular highlight distribution as moisture indicator

## Architecture
- Backbone: EfficientNet-B0 (pretrained ImageNet)
- Feature fusion: CNN features + handcrafted Gabor/LBP features
- Regression head for hydration score [0-100]

## Output
- ONNX model for backend inference

In [ ]:
# Install dependencies (Colab)
!pip install -q torch torchvision timm albumentations onnx onnxruntime opencv-python scikit-image

In [ ]:
import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Feature Extraction

Gabor filters capture texture at multiple orientations and frequencies. LBP captures
micro-texture patterns. Specular highlights indicate skin moisture level.

In [ ]:
def build_gabor_bank(orientations: int = 4, frequencies: list = [0.1, 0.25, 0.4]) -> list:
    """Build Gabor filter bank: 4 orientations x 3 frequencies = 12 filters."""
    kernels = []
    for theta_idx in range(orientations):
        theta = theta_idx * np.pi / orientations
        for freq in frequencies:
            kernel = cv2.getGaborKernel(
                ksize=(31, 31), sigma=4.0, theta=theta,
                lambd=1.0 / freq, gamma=0.5, psi=0
            )
            kernels.append(kernel)
    return kernels


def extract_gabor_features(gray: np.ndarray, kernels: list) -> np.ndarray:
    """Apply Gabor bank and compute mean + std energy per filter."""
    features = []
    for kernel in kernels:
        response = cv2.filter2D(gray, cv2.CV_64F, kernel)
        features.extend([response.mean(), response.std()])
    return np.array(features, dtype=np.float32)


def extract_lbp_histogram(gray: np.ndarray, radius: int = 2, n_points: int = 16) -> np.ndarray:
    """Compute uniform LBP histogram."""
    lbp = local_binary_pattern(gray, n_points, radius, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=n_points + 2, range=(0, n_points + 2), density=True)
    return hist.astype(np.float32)


def extract_specular_features(image: np.ndarray, threshold: int = 220) -> np.ndarray:
    """Analyze specular highlight distribution as moisture indicator."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    highlights = (gray > threshold).astype(np.float32)
    highlight_ratio = highlights.mean()
    # Spatial spread of highlights
    if highlights.sum() > 0:
        coords = np.argwhere(highlights > 0)
        spread = coords.std(axis=0).mean() / max(gray.shape)
    else:
        spread = 0.0
    return np.array([highlight_ratio, spread], dtype=np.float32)


GABOR_BANK = build_gabor_bank()
HANDCRAFTED_DIM = len(GABOR_BANK) * 2 + 18 + 2  # gabor(24) + lbp(18) + specular(2) = 44
print(f"Handcrafted feature dimension: {HANDCRAFTED_DIM}")

## Dataset

In [ ]:
class HydrationDataset(Dataset):
    """Dataset for hydration signal training."""

    def __init__(self, image_dir: str, annotations_path: str, transform=None):
        self.image_dir = image_dir
        with open(annotations_path) as f:
            self.annotations = json.load(f)
        self.image_ids = list(self.annotations.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        ann = self.annotations[img_id]

        image = cv2.imread(os.path.join(self.image_dir, img_id))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image, (224, 224))

        # Handcrafted features
        gray = cv2.cvtColor(image_resized, cv2.COLOR_RGB2GRAY)
        gabor_feat = extract_gabor_features(gray, GABOR_BANK)
        lbp_feat = extract_lbp_histogram(gray)
        specular_feat = extract_specular_features(image_resized)
        handcrafted = np.concatenate([gabor_feat, lbp_feat, specular_feat])

        if self.transform:
            augmented = self.transform(image=image_resized)
            image_tensor = augmented["image"]
        else:
            image_tensor = torch.from_numpy(image_resized.transpose(2, 0, 1)).float() / 255.0

        label = torch.tensor([ann["hydration_score"]], dtype=torch.float32)
        return image_tensor, torch.from_numpy(handcrafted), label


train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

## Model Definition

EfficientNet-B0 backbone with feature fusion. CNN features are concatenated with
handcrafted Gabor/LBP/specular features before the regression head.

In [ ]:
class HydrationModel(nn.Module):
    """Hydration signal model with EfficientNet-B0 + handcrafted feature fusion."""

    def __init__(self, handcrafted_dim: int = 44, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=0)
        cnn_dim = self.backbone.num_features  # 1280

        self.handcrafted_fc = nn.Sequential(
            nn.Linear(handcrafted_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.head = nn.Sequential(
            nn.Linear(cnn_dim + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, image, handcrafted):
        cnn_feat = self.backbone(image)
        hc_feat = self.handcrafted_fc(handcrafted)
        fused = torch.cat([cnn_feat, hc_feat], dim=-1)
        return self.head(fused)


model = HydrationModel(handcrafted_dim=HANDCRAFTED_DIM, pretrained=True).to(DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Loop

In [ ]:
IMAGE_DIR = "./data/hydration/images"
ANNOTATIONS_PATH = "./data/hydration/annotations.json"

NUM_EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-4


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for images, handcrafted, labels in tqdm(loader, desc="Training"):
        images = images.to(DEVICE)
        handcrafted = handcrafted.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        preds = model(images, handcrafted)
        loss = nn.functional.mse_loss(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, handcrafted, labels in loader:
        images, handcrafted = images.to(DEVICE), handcrafted.to(DEVICE)
        preds = model(images, handcrafted)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
    preds = torch.cat(all_preds).squeeze()
    labels = torch.cat(all_labels).squeeze()
    mae = torch.abs(preds - labels).mean().item()
    pearson = torch.corrcoef(torch.stack([preds, labels]))[0, 1].item()
    return {"mae": mae, "pearson_r": pearson}


# Training (uncomment when dataset is available)
# train_ds = HydrationDataset(IMAGE_DIR, ANNOTATIONS_PATH, transform=train_transform)
# val_ds = HydrationDataset(IMAGE_DIR, ANNOTATIONS_PATH, transform=val_transform)
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# best_mae = float("inf")
# for epoch in range(NUM_EPOCHS):
#     train_loss = train_one_epoch(model, train_loader, optimizer)
#     metrics = evaluate(model, val_loader)
#     scheduler.step()
#     print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
#           f"MAE: {metrics['mae']:.2f} | Pearson r: {metrics['pearson_r']:.4f}")
#     if metrics["mae"] < best_mae:
#         best_mae = metrics["mae"]
#         torch.save(model.state_dict(), "best_hydration_model.pt")
#         print(f"  -> Saved best model (MAE: {best_mae:.2f})")

## ONNX Export

In [ ]:
def export_to_onnx(model, output_path="hydration_model.onnx"):
    model.eval()
    dummy_image = torch.randn(1, 3, 224, 224).to(DEVICE)
    dummy_hc = torch.randn(1, HANDCRAFTED_DIM).to(DEVICE)

    torch.onnx.export(
        model,
        (dummy_image, dummy_hc),
        output_path,
        input_names=["image", "handcrafted_features"],
        output_names=["hydration_score"],
        dynamic_axes={
            "image": {0: "batch"},
            "handcrafted_features": {0: "batch"},
            "hydration_score": {0: "batch"},
        },
        opset_version=17,
    )
    print(f"Exported ONNX model to {output_path}")

    # Verify
    import onnxruntime as ort
    session = ort.InferenceSession(output_path)
    result = session.run(None, {
        "image": dummy_image.cpu().numpy(),
        "handcrafted_features": dummy_hc.cpu().numpy(),
    })
    print(f"ONNX output shape: {result[0].shape}, sample: {result[0][0][0]:.2f}")


# export_to_onnx(model)